In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

IN = Path("../data/diabetes_clean.csv")
PROC = Path("../data/processed")
PROC.mkdir(parents=True, exist_ok=True)

In [2]:
id_cols=["admission_type_id","discharge_disposition_id","admission_source_id"]

df=pd.read_csv(IN,keep_default_na=False,dtype={c: str for c in id_cols})

print(df.shape)
print(df[["max_glu_serum","A1Cresult"]].apply(lambda s: s.value_counts()).fillna(0))

(69987, 47)
      max_glu_serum  A1Cresult
>200          936.0        0.0
>300          712.0        0.0
>7              0.0     2866.0
>8              0.0     6239.0
None        66638.0    57141.0
Norm         1701.0     3741.0


In [3]:
def icd9_group(code):
    if code=="Missing":
        return "Missing"
    if code.startswith(("V","E")):      
        return "Other"
    n=float(code)
    if 250<=n< 251:                  
        return "Diabetes"
    if 390<=n<460 or n==785:      
        return "Circulatory"
    if 460<=n<520 or n==786:       
        return "Respiratory"
    if 520<=n<580 or n==787:       
        return "Digestive"
    if 580<=n<630 or n==788:      
        return "Genitourinary"
    if 140<=n<240:                   
        return "Neoplasms"
    if 710<=n<740:                   
        return "Musculoskeletal"
    if 800<=n<1000:                  
        return "Injury"
    return "Other"

for i in [1,2,3]:
    df[f"diag_{i}_group"]=df[f"diag_{i}"].apply(icd9_group)

df["diag_1_group"].value_counts()

diag_1_group
Circulatory        21389
Other              12124
Respiratory         9491
Digestive           6488
Diabetes            5748
Injury              4694
Musculoskeletal     4064
Genitourinary       3441
Neoplasms           2538
Missing               10
Name: count, dtype: int64

In [4]:
df["total_prior_visits"]=(df["number_outpatient"]+df["number_emergency"]+df["number_inpatient"])

df["any_prior_inpatient"]=(df["number_inpatient"]>0).astype(int)

In [5]:
med_cols=["metformin","repaglinide","nateglinide","chlorpropamide",
            "glimepiride", "acetohexamide","glipizide","glyburide",
            "tolbutamide","pioglitazone","rosiglitazone","acarbose",
            "miglitol","troglitazone","tolazamide","insulin",
            "glyburide-metformin","glipizide-metformin",
            "glimepiride-pioglitazone","metformin-rosiglitazone",
            "metformin-pioglitazone"]

df["n_diabetes_meds"]=(df[med_cols]!="No").sum(axis=1)
df["n_meds_changed"]=df[med_cols].isin(["Up","Down"]).sum(axis=1)

df[["n_diabetes_meds","n_meds_changed"]].describe().T

,count,mean,std,min,25%,50%,75%,max
n_diabetes_meds,69987.0,1.190078,0.942556,0.0,1.0,1.0,2.0,6.0
n_meds_changed,69987.0,0.261863,0.475830,0.0,0.0,0.0,0.0,4.0


In [6]:
df["meds_per_day"]=df["num_medications"]/df["time_in_hospital"]
df["procedures_total"]=df["num_lab_procedures"]+df["num_procedures"]

In [7]:
target="readmit_30"

drop_from_features=["readmitted","age","diag_1","diag_2","diag_3"]

X=df.drop(columns=drop_from_features+[target])
y=df[target]

print(X.shape)
print("Categorical:",X.select_dtypes("object").columns.tolist())
print("Numeric:",X.select_dtypes("number").columns.tolist())

(69987, 50)
Categorical: ['race', 'gender', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'payer_code', 'medical_specialty', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'diag_1_group', 'diag_2_group', 'diag_3_group']
Numeric: ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'age_mid', 'total_prior_visits', 'any_prior_inpatient', 'n_diabetes_meds', 'n_meds_changed', 'meds_per_day', 'procedures_total']


C:\Users\hp\AppData\Local\Temp\ipykernel_20632\2659052403.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print("Categorical:",X.select_dtypes("object").columns.tolist())


In [8]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)

print("Train:",X_train.shape," Test:",X_test.shape)
print("Positive rate train:",y_train.mean().round(4))
print("Positive rate test: ",y_test.mean().round(4))

Train: (55989, 50)  Test: (13998, 50)
Positive rate train: 0.0898
Positive rate test:  0.0898


In [9]:
train=X_train.assign(**{target:y_train})
test=X_test.assign(**{target:y_test})

train.to_csv(PROC/"train.csv",index=False)
test.to_csv(PROC/"test.csv",index=False)
print("Saved train and test to",PROC)

Saved train and test to ..\data\processed


## Feature engineering summary
- ICD-9 codes grouped into ~10 body-system categories for diag_1/2/3.
- Added total_prior_visits, any_prior_inpatient, n_diabetes_meds,
  n_meds_changed, meds_per_day, procedures_total.
- Excluded from features: readmitted (target source), age, raw diag codes.
- Stratified 80/20 split, random_state=42; test set untouched until final evaluation.
- Not done here on purpose: encoding, scaling, grouping rare categories
  (e.g. medical_specialty), and resampling. These learn from the data,
  so they go inside the training pipeline to avoid leakage.